# Run the 9-balancing-area model

Choose `RUN_MODE`, `UNIT_COMMITMENT`, and a unique `RUN_NAME`, edit the settings relevant to that mode, then run all cells. Results are stored under `results/run_results/<mode>/<run_name>/`; reusing a name resumes or overwrites that named run.

- `daily`: runs the date in `REFERENCE_DATE` (24 hours).
- `monthly`: runs every day in the month containing `REFERENCE_DATE`. To keep unit commitment tractable, each day is solved independently.
- `peak`: runs one continuous 168-hour unit-commitment problem for the week containing the annual peak demand. `REFERENCE_DATE` is ignored.
- `selected_weeks`: runs each date in `WEEK_STARTS` as an independent continuous seven-day problem, with configurable warm-up and look-ahead days.
- `rolling`: runs the full 8,760-hour year as rolling windows. Each solve optimizes seven committed days plus one look-ahead day, carries commitment/dispatch/storage state forward, and checkpoints every completed week. `REFERENCE_DATE` is ignored.

`UNIT_COMMITMENT = "full"` uses binary on/off decisions; `"relaxed"` uses continuous on/off fractions and is faster but approximate; `"none"` removes on/off decisions and minimum stable output from formerly committable generators. In rolling mode, fractional statuses at a window boundary are rounded for state transfer. Non-full modes save to separate result folders.

Gas and nuclear daily energy are equality constrained in every mode. Results are saved automatically under `results/run_results/`. Rolling mode resumes existing completed checkpoints when `RESUME = True`.

After a rolling run, use `9BA_results_analysis_rolling_year.ipynb` for the full-year analysis, including wind-resource and hydro-utilization seasonality against observed FY2025-26 generation.

In [ ]:
# ===================== USER SETTINGS =====================
RUN_MODE = "rolling"   # "daily", "monthly", "peak", "selected_weeks", or "rolling"
REFERENCE_DATE = "2025-07-11"  # day; month is inferred; ignored for peak/selected_weeks/rolling
WEEK_STARTS = [
    "2025-04-07",  # dry / low hydro
    "2025-07-07",  # southwest-monsoon wind and hydro peak
    "2025-10-13",  # monsoon transition
    "2026-02-16",  # dry-season low renewables
]
WARMUP_DAYS = 1
LOOKAHEAD_DAYS = 1
RESERVOIR_INITIAL_SOC_FRACTION = 0.0  # independent-week assumption; test sensitivity if needed
SOLVER_NAME = "highs"
MIP_REL_GAP = 0.01            # 1% gap; increase for a faster approximate solve
UNIT_COMMITMENT = "relaxed"    # "full", "relaxed", or "none"
RUN_NAME = "hydro_relaxed"    # choose a new name to keep each run separate
ROLLING_TIME_LIMIT = 1800     # seconds per rolling window; set None for no cap
ROLLING_MAX_TIME_LIMIT_MIP_GAP = 0.05  # accept a capped solve only up to 5% gap
RESUME = False                  # selected_weeks/rolling: reuse compatible completed solves
# =========================================================

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
if not (ROOT / "scripts").exists():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks folder.")

sys.path.insert(0, str(ROOT / "scripts"))
from run_9ba_selection import run_selection

result = run_selection(
    RUN_MODE,
    REFERENCE_DATE,
    week_starts=WEEK_STARTS,
    warmup_days=WARMUP_DAYS,
    lookahead_days=LOOKAHEAD_DAYS,
    reservoir_initial_soc_fraction=RESERVOIR_INITIAL_SOC_FRACTION,
    solver_name=SOLVER_NAME,
    mip_rel_gap=MIP_REL_GAP,
    unit_commitment=UNIT_COMMITMENT,
    run_name=RUN_NAME,
    rolling_time_limit=ROLLING_TIME_LIMIT,
    rolling_max_time_limit_mip_gap=ROLLING_MAX_TIME_LIMIT_MIP_GAP,
    resume=RESUME,
)

In [ ]:
from IPython.display import Image, display
import pandas as pd

print(f"Mode: {result['mode']}")
print(f"Period: {result['dates'][0].date()} to {result['dates'][-1].date()}")
print(f"Saved to: {result['output_dir']}")
if result.get('week_dirs'):
    print('Per-week results:')
    for path in result['week_dirs']:
        print(f'  {path}')
display(result["summary"].round(3))
display(pd.Series(result["validation"], name="value").to_frame())
display(Image(filename=str(result["plot_path"])))